# Actualización a ENIGH 2022 — Aradillas (2018)
## Poder de mercado y bienestar social en hogares mexicanos

Aplica a la ENIGH 2022 la metodología del estudio de COFECE replicada en
`aradillas_2014.ipynb`. El cálculo vive en `aradillas_core.py` y la carga de datos
en `datos_2022.py`; este notebook solo los conecta.

Ejecutar desde el directorio que contiene `Replica_COFECE/`.

---

### Desviación deliberada: se corre SIN el trim iterado

El Gauss recorta el 1 % de cada cola de la utilidad **en cada una de las 16 iteraciones**
del bucle OLS, de forma acumulativa. En 2014 eso cuesta el 28 % de la muestra y no hace
daño. **En 2022 destruye la identificación**, y la razón es una interacción entre el trim y
el tamaño de muestra: con 8,940 hogares las colas de la utilidad se regeneran entre
iteraciones y su varianza sobrevive; con 57,552 los cuantiles son estables, el recorte
muerde siempre en el mismo lugar y la varianza colapsa.

| | 2014 con trim | 2022 con trim | 2022 sin trim |
|---|---|---|---|
| varianza de `util` en el bucle | 1.170 | **0.450** | 0.798 |
| hogares con `f'(u0)` plano (`\|f'\|<0.3`) | 6.9 % | **76.1 %** | 33.6 % |
| distancia del 1er paso de Newton | 1.13 | **14.48** | 6.16 |
| convergencia del solver | ~97 % | **31.9 %** | **88.4 %** |
| rango de elasticidades | — | **[0.042, 3.347]** | **[0.418, 1.756]** |

La cadena: el trim aplana la varianza del regresor `util` → los coeficientes en `u`
(los efectos ingreso) quedan mal identificados → la función de costo es casi plana donde
arranca Newton → la raíz cae lejísimos → utilidades en `[−73, 107]` → elasticidades
inservibles.

**Por qué desviarse es lo correcto aquí:** el criterio de fidelidad al Gauss aplica a la
*réplica* de 2014, donde hay un programa original contra el cual contrastar. La
actualización a 2022 es trabajo nuevo, y conservar un mecanismo que demostrablemente
destruye la identificación en esta muestra no sería fidelidad. **Queda documentado como
hallazgo: el algoritmo del estudio original no escala a muestras del tamaño de la ENIGH
moderna.**

### Diferencias de datos respecto de 2014

* Hogar = `folioviv` + `foliohog` (una vivienda puede alojar varios hogares).
* Claves de producto alfanuméricas (`A004`) en vez de numéricas (`1004`). La conversión es
  determinista y las categorías se derivan de la tabla verificada de `datos_2014.py`, lo
  que evita el error de `productos.py` que ponía el aguacate en Frutas y omitía la pera.
* Precios de referencia de **2018**, deflactados a la ventana ago–nov 2022 (factor mediano
  1.367). Materiales va por INPP, que le da la variación entre ciudades.
* **`ing_mon` no existe**: la ENIGH 2022 "Nueva serie" dejó de publicar el ingreso no
  monetario por separado. Se reconstruye sumando `ingresos.csv`, que **ya contiene solo
  ingreso monetario** — el componente no monetario (`estim_alqu`) es imputado y no aparece
  ahí. Verificado: Σ claves = \$53,929 ≈ `ing_cor − estim_alqu − remu_espec` = \$53,694.

In [1]:
import sys
sys.path.insert(0, "Replica_COFECE/Codigo")
import numpy as np

import datos_2022
from aradillas_core import (estimar_easi, reconstruir_matrices, ModeloEASI,
                            demandas_marshallianas, elasticidades,
                            estimar_markups, variacion_equivalente,
                            cuadro_10, gini, sectores_significativos)

DATA_DIR = "Replica_COFECE/Data_2022/"
print("Módulos cargados.")

Módulos cargados.


## 1–2. Precios locales y microdatos ENIGH 2022

Cascada esperada: 90,102 → 57,989 (filtros del Gauss) → 57,989 (400 km) → 57,552
(al menos una categoría con gasto ≥ 10).

El filtro de 400 km no descarta a nadie, y **no es un error**: la ENIGH 2022 cubre 90 mil
hogares con mucha mayor dispersión geográfica y prácticamente todos caen dentro de ese
radio de alguna de las 46 ciudades. En 2014 ese filtro solo quitaba 131 de 12,592.

In [2]:
datos = datos_2022.cargar(DATA_DIR)

ing_mon reconstruido (corte P999): razón ing_mon/ing_cor = 0.877   [2014: 0.794]


/Users/benjamin/GAMES Econ Dropbox/Benjamín  Oliva/GAMESEcon Economists CDMX/01 Data/Data_Aradillas/Replica_COFECE/Codigo/datos_2022.py:381: DtypeWarning: Columns (4,24) have mixed types. Specify dtype option on import or set low_memory=False.
  viv = pd.read_csv(data_dir + 'viviendas.csv', dtype={'folioviv': str})


Hogares después de filtros básicos: 57989


Coordenadas resueltas para las 46 ciudades INPC.
Hogares después de filtro de distancia (<=400 km): 57989


/Users/benjamin/GAMES Econ Dropbox/Benjamín  Oliva/GAMESEcon Economists CDMX/01 Data/Data_Aradillas/Replica_COFECE/Codigo/datos_2022.py:263: RuntimeWarning: All-NaN slice encountered
  b = np.nanmedian(serie[base])
/Users/benjamin/GAMES Econ Dropbox/Benjamín  Oliva/GAMESEcon Economists CDMX/01 Data/Data_Aradillas/Replica_COFECE/Codigo/datos_2022.py:263: RuntimeWarning: All-NaN slice encountered
  b = np.nanmedian(serie[base])
/Users/benjamin/GAMES Econ Dropbox/Benjamín  Oliva/GAMESEcon Economists CDMX/01 Data/Data_Aradillas/Replica_COFECE/Codigo/datos_2022.py:263: RuntimeWarning: All-NaN slice encountered
  b = np.nanmedian(serie[base])
/Users/benjamin/GAMES Econ Dropbox/Benjamín  Oliva/GAMESEcon Economists CDMX/01 Data/Data_Aradillas/Replica_COFECE/Codigo/datos_2022.py:263: RuntimeWarning: All-NaN slice encountered
  b = np.nanmedian(serie[base])
/Users/benjamin/GAMES Econ Dropbox/Benjamín  Oliva/GAMESEcon Economists CDMX/01 Data/Data_Aradillas/Replica_COFECE/Codigo/datos_2022.py

/Users/benjamin/GAMES Econ Dropbox/Benjamín  Oliva/GAMESEcon Economists CDMX/01 Data/Data_Aradillas/Replica_COFECE/Codigo/datos_2022.py:263: RuntimeWarning: All-NaN slice encountered
  b = np.nanmedian(serie[base])
/Users/benjamin/GAMES Econ Dropbox/Benjamín  Oliva/GAMESEcon Economists CDMX/01 Data/Data_Aradillas/Replica_COFECE/Codigo/datos_2022.py:263: RuntimeWarning: All-NaN slice encountered
  b = np.nanmedian(serie[base])
/Users/benjamin/GAMES Econ Dropbox/Benjamín  Oliva/GAMESEcon Economists CDMX/01 Data/Data_Aradillas/Replica_COFECE/Codigo/datos_2022.py:263: RuntimeWarning: All-NaN slice encountered
  b = np.nanmedian(serie[base])
/Users/benjamin/GAMES Econ Dropbox/Benjamín  Oliva/GAMESEcon Economists CDMX/01 Data/Data_Aradillas/Replica_COFECE/Codigo/datos_2022.py:263: RuntimeWarning: All-NaN slice encountered
  b = np.nanmedian(serie[base])
/Users/benjamin/GAMES Econ Dropbox/Benjamín  Oliva/GAMESEcon Economists CDMX/01 Data/Data_Aradillas/Replica_COFECE/Codigo/datos_2022.py

/Users/benjamin/GAMES Econ Dropbox/Benjamín  Oliva/GAMESEcon Economists CDMX/01 Data/Data_Aradillas/Replica_COFECE/Codigo/datos_2022.py:263: RuntimeWarning: All-NaN slice encountered
  b = np.nanmedian(serie[base])
/Users/benjamin/GAMES Econ Dropbox/Benjamín  Oliva/GAMESEcon Economists CDMX/01 Data/Data_Aradillas/Replica_COFECE/Codigo/datos_2022.py:263: RuntimeWarning: All-NaN slice encountered
  b = np.nanmedian(serie[base])
/Users/benjamin/GAMES Econ Dropbox/Benjamín  Oliva/GAMESEcon Economists CDMX/01 Data/Data_Aradillas/Replica_COFECE/Codigo/datos_2022.py:263: RuntimeWarning: All-NaN slice encountered
  b = np.nanmedian(serie[base])
/Users/benjamin/GAMES Econ Dropbox/Benjamín  Oliva/GAMESEcon Economists CDMX/01 Data/Data_Aradillas/Replica_COFECE/Codigo/datos_2022.py:263: RuntimeWarning: All-NaN slice encountered
  b = np.nanmedian(serie[base])
/Users/benjamin/GAMES Econ Dropbox/Benjamín  Oliva/GAMESEcon Economists CDMX/01 Data/Data_Aradillas/Replica_COFECE/Codigo/datos_2022.py

  sin serie INPC (1): ['Área Met. de la Cd. de México']...
  factores de deflactación 2018->2022: mediana=1.367 [0.67, 3.03]  (4646 pares)
  materiales (INPP): [122.8, 153.0]  (4 ciudades imputadas)


Hogares finales en la muestra: 57552


ENIGH 2022: 57552 hogares, 12 categorías, 46 ciudades


## 4–5. Sistema EASI y utilidad indirecta

`aplicar_trim=False` — ver la justificación en la portada.

In [3]:
res = estimar_easi(datos.precios_ln, datos.w, datos.gasto_total, datos.Z,
                   n_cat=datos.n_cat, aplicar_trim=False)
datos = datos.submuestra(res["mask"])

mats = reconstruir_matrices(res["beta"], n_cat=datos.n_cat)
modelo = ModeloEASI(**mats)
epsilon = res["epsilon"]

util = modelo.utilidad_indirecta(datos.precios_ln, datos.Z, epsilon,
                                 datos.w, datos.gasto_total)

  Iteración  1: (primera estimación)  N=57552


  Iteración  2: criterio = 0.907385  N=57552


  Iteración  3: criterio = 0.165220  N=57552


  Iteración  4: criterio = 0.060885  N=57552


  Iteración  5: criterio = 0.020705  N=57552


...

  18000/57552  (fallbacks: 1658)
  20000/57552  (fallbacks: 1884)
  22000/57552  (fallbacks: 2179)
  24000/57552  (fallbacks: 2490)
  26000/57552  (fallbacks: 2866)


  28000/57552  (fallbacks: 3144)
  30000/57552  (fallbacks: 3498)
  32000/57552  (fallbacks: 3799)
  34000/57552  (fallbacks: 4055)
  36000/57552  (fallbacks: 4338)
  38000/57552  (fallbacks: 4606)
  40000/57552  (fallbacks: 4839)


  42000/57552  (fallbacks: 4960)
  44000/57552  (fallbacks: 5244)
  46000/57552  (fallbacks: 5484)
  48000/57552  (fallbacks: 5639)


  50000/57552  (fallbacks: 5883)
  52000/57552  (fallbacks: 6261)
  54000/57552  (fallbacks: 6439)
  56000/57552  (fallbacks: 6554)

Newton converge: 50854/57552 (88.4%)


Utilidad: media=0.7706  std=2.9485  rango [-6.247, 18.960]


## 6. Elasticidades

In [4]:
demandas, _ = demandas_marshallianas(
    modelo, datos.precios_ln, datos.Z, util, epsilon,
    datos.gasto_total, datos.factor_expansion, datos.n_cat)

elastic_nac, elastic_cd = elasticidades(
    modelo, datos.precios_ln, datos.Z, epsilon, datos.w, datos.gasto_total,
    datos.factor_expansion, demandas, datos.ciudad, datos.n_ciudades,
    datos.n_cat, util, nombres=datos.nombres_cat)

  [ 1] Tortillas              

e=0.803
  [ 2] Pan                    

e=1.391
  [ 3] Pollo+Huevo            

e=1.430
  [ 4] Carne res              

e=0.612
  [ 5] Carnes proc.           

...

e=1.278
  [ 9] Bebidas                

e=1.756
  [10] Medicamentos           

e=0.687
  [11] Transporte foráneo     

e=0.418
  [12] Materiales             

e=0.887


In [5]:
paper14 = {'Tortillas':1.054,'Pan':1.462,'Pollo+Huevo':1.261,'Carne res':0.735,
           'Carnes proc.':0.968,'Lácteos':1.289,'Frutas':1.415,'Verduras':1.389,
           'Bebidas':1.110,'Medicamentos':0.943,'Transporte foráneo':0.847,
           'Materiales':0.934}

print("=== ELASTICIDADES 2022 ===")
print(f"{'Categoría':<22} {'2022':>8} {'paper 2014':>11} {'dif':>8}")
print("-"*52)
for j, n in enumerate(datos.nombres_cat):
    e = abs(elastic_nac[j]); p = paper14[n]
    print(f"  {n:<20} {e:>8.3f} {p:>11.3f} {e-p:>8.3f}")
print(f"\nrango: [{np.abs(elastic_nac).min():.3f}, {np.abs(elastic_nac).max():.3f}]"
      f"   (paper 2014: [0.735, 1.462])")

=== ELASTICIDADES 2022 ===
Categoría                  2022  paper 2014      dif
----------------------------------------------------
  Tortillas               0.803       1.054   -0.251
  Pan                     1.391       1.462   -0.071
  Pollo+Huevo             1.430       1.261    0.169
  Carne res               0.612       0.735   -0.123
  Carnes proc.            1.113       0.968    0.145
  Lácteos                 1.249       1.289   -0.040
  Frutas                  1.457       1.415    0.042
  Verduras                1.278       1.389   -0.111
  Bebidas                 1.756       1.110    0.646
  Medicamentos            0.687       0.943   -0.256
  Transporte foráneo      0.418       0.847   -0.429
  Materiales              0.887       0.934   -0.047

rango: [0.418, 1.756]   (paper 2014: [0.735, 1.462])


## 7. Markups y poder de mercado (NEIO)

**Cómo leer el cuadro.** Si las elasticidades estuvieran comprimidas hacia −1, entonces
η ≈ p y la regresión devolvería β ≈ 1 con estadísticos t enormes por colinealidad casi
perfecta — una identidad algebraica, no poder de mercado. Que los β caigan en rango
plausible, que los t sean moderados y que **algunos sectores salgan no significativos** es
la señal de que la estimación es sana.

In [6]:
P_cat = datos.precios_por_ciudad()
mk = estimar_markups(P_cat, elastic_cd, datos.vars_costos)
beta_eta, t_eta = mk["beta_eta"], mk["t_eta"]

paper_b = {'Tortillas':0.183,'Pan':1.477,'Pollo+Huevo':0.139,'Carne res':0.047,
           'Carnes proc.':0.017,'Lácteos':0.626,'Frutas':1.120,'Verduras':0.328,
           'Bebidas':0.047,'Medicamentos':0.026,'Transporte foráneo':0.081,
           'Materiales':0.493}

print("=== CUADRO 8: PODER DE MERCADO β_η ===")
print(f"{'Categoría':<22} {'β 2022':>8} {'t':>8} {'β paper14':>10}")
print("-"*52)
for j, n in enumerate(datos.nombres_cat):
    sig = "***" if abs(t_eta[j]) >= 2.326 else ("**" if abs(t_eta[j]) >= 1.645 else "   ")
    print(f"  {n:<20} {beta_eta[j]:>8.3f} {t_eta[j]:>8.2f} {paper_b[n]:>10.3f} {sig}")

=== CUADRO 8: PODER DE MERCADO β_η ===
Categoría                β 2022        t  β paper14
----------------------------------------------------
  Tortillas               0.349     3.63      0.183 ***
  Pan                     1.496    19.06      1.477 ***
  Pollo+Huevo             0.095     1.56      0.139    
  Carne res               0.105     3.14      0.047 ***
  Carnes proc.            0.322     5.38      0.017 ***
  Lácteos                 0.084     1.06      0.626    
  Frutas                  0.685     3.90      1.120 ***
  Verduras                0.136     2.12      0.328 **
  Bebidas                 0.442     3.68      0.047 ***
  Medicamentos            0.406     5.02      0.026 ***
  Transporte foráneo      0.013     0.18      0.081    
  Materiales              0.515     5.64      0.493 ***


## 8. Variación equivalente y bienestar

El denominador es el ingreso monetario reconstruido desde `ingresos.csv`. El Gini se
calcula como en el Gauss: sobre la muestra completa del concentrado y con ajuste
multiplicativo por decil.

**Advertencia sobre magnitudes en pesos.** La VE de 2022 equivale al 77 % del gasto en las
doce categorías, contra ~53 % en 2014. Puede ser real —9 de 12 sectores resultan
significativos, con markups mayores— o indicar sobreestimación. **Revisar antes de publicar
cifras en pesos**; las elasticidades y los β_η no dependen de esto.

El Gini observado de 2022 **no es directamente comparable** con el de 2014: son bases de
ingreso distintas, no desigualdades distintas.

In [7]:
sig_95 = sectores_significativos(t_eta, beta_eta)   # V1: t >= 2.326
print(f"Sectores significativos: {int(sig_95.sum())} de {datos.n_cat}")

VE = variacion_equivalente(modelo, datos.precios_ln, datos.Z, epsilon, datos.w,
                           datos.gasto_total,
                           mk["markup_lerner"][datos.ciudad],   # V2: Lerner
                           sig_95)
c10 = cuadro_10(VE, datos.ingreso_cor)  # V3: ing_cor — la ENIGH 2022 "Nueva serie" ya no publica ing_total

paper_p = [30.9,23.6,21.4,18.9,16.7,15.1,13.6,11.9,9.5,5.7,15.7]
print("\n=== CUADRO 10: PÉRDIDA DE BIENESTAR POR DECIL ===")
print(f"{'Decil':<6} {'VE($)':>9} {'VE/Ing%':>9} {'paper14%':>9}")
print("-"*38)
for f in c10["deciles"]:
    print(f"  {f['decil']:<4} {f['VE']:>9.0f} {f['pct']:>9.1f} {paper_p[f['decil']-1]:>9.1f}")
print(f"  {'Tot':<4} {c10['total']['VE']:>9.0f} {c10['total']['pct']:>9.1f} {paper_p[-1]:>9.1f}")
print(f"\nRegresividad D1/D10: {c10['regresividad']:.2f}  (paper 2014: 4.42)")

g = gini(datos.ingreso_mon_completo, c10["tasas"])
print(f"\nGini observado:     {g['observado']:.3f}   [N={g['n']}]")
print(f"Gini contrafactual: {g['contrafactual']:.3f}")
print(f"Reducción:          {g['reduccion_pct']:.1f}%  (paper 2014: 7.3%)")

Sectores significativos: 8 de 12
  0/57552...


  2000/57552...


  4000/57552...


  6000/57552...


  8000/57552...


...

  50000/57552...


  52000/57552...


  54000/57552...


  56000/57552...



=== CUADRO 10: PÉRDIDA DE BIENESTAR POR DECIL ===
Decil      VE($)   VE/Ing%  paper14%
--------------------------------------
  1         1294      10.0      30.9
  2         1836       8.1      23.6
  3         2232       7.5      21.4
  4         2537       6.9      18.9
  5         2813       6.3      16.7
  6         3022       5.6      15.1
  7         3274       5.1      13.6
  8         3529       4.5      11.9
  9         3751       3.7       9.5
  10        4082       2.3       5.7
  Tot       2711       5.4      15.7

Regresividad D1/D10: 4.39  (paper 2014: 4.42)

Gini observado:     0.446   [N=90044]
Gini contrafactual: 0.436
Reducción:          2.2%  (paper 2014: 7.3%)
